### Step 1: Text Extraction -> Done externally


### Step 2: Tokenization

In [2]:
import torch
# device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
import numpy as np

with open('..//..//data//processed//combined_texts.txt', 'r', encoding='utf-8') as f:
    text = f.read()

char_set= set(text)
chars_sorted= sorted(char_set)

char2int= {ch:i for i, ch in enumerate(chars_sorted)}
char_array= np.array(chars_sorted)
text_encoded= np.array([char2int[ch] for ch in text], dtype=np.int32)


In [4]:
len(char_array), len(text_encoded)

(149, 7508154)

### Set The dataset architecture

In [6]:
from torch.utils.data import Dataset, DataLoader

SEQ_LEN = 128
chunk_size = SEQ_LEN + 1
text_chunks = [text_encoded[i:i + chunk_size] for i in range(0, len(text_encoded) - chunk_size)]

class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks

    def __len__(self):
        return len(self.text_chunks)

    def __getitem__(self, idx):
        chunk = self.text_chunks[idx]
        input_seq = torch.tensor(chunk[:-1], dtype=torch.long).to(device)
        target_seq = torch.tensor(chunk[1:], dtype=torch.long).to(device)
        return input_seq, target_seq

seq_dataset = TextDataset(text_chunks)

In [7]:
# Let's set the dataloader
BATCH_SIZE = 64  #hyperparameter
torch.manual_seed(42)
seq_dl= DataLoader(seq_dataset, batch_size=BATCH_SIZE, shuffle=True)


### Model: Transformer Encoder only

### Model: decoder-only Transformer (GPT-style)

For text generation, the model must be causal: at position `t`, it can see characters `0` through `t`, but never future characters. Each training target is the input sequence shifted one character to the left. This is next-token prediction, the central training objective behind GPT models.

In [8]:
from torch.utils.data import random_split

torch.manual_seed(42)

train_dataset, val_dataset= random_split(seq_dataset, [0.9,0.1])

BATCH_SIZE= 64
train_dl= DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_dl= DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
# Note: drop_last=True is used to ensure that all batches are of the same size, which is important for training stability.(exp: BatchNorm, etc.)

vocab_size = len(char_array)
print(f"Vocabulary: {vocab_size} characters")
print(f"Training sequences: {len(train_dataset):,}")
print(f"Validation sequences: {len(val_dataset):,}")

Vocabulary: 149 characters
Training sequences: 6,757,223
Validation sequences: 750,802


In [9]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class CharacterGPT(nn.Module):
    def __init__(self, vocab_size, seq_len, embedding_dim=256, num_layers=4, num_heads=4, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(seq_len, embedding_dim)

        block = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=embedding_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(block, num_layers=num_layers)
        self.final_norm = nn.LayerNorm(embedding_dim)
        self.lm_head = nn.Linear(embedding_dim, vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding.weight
        nn.init.normal_(self.token_embedding.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.position_embedding.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        _, sequence_length = input_ids.shape
        positions = torch.arange(sequence_length, device=input_ids.device)
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions)

        causal_mask = torch.triu(
            torch.ones(sequence_length, sequence_length, device=input_ids.device, dtype=torch.bool),
            diagonal=1,
        )
        hidden = self.transformer(hidden, mask=causal_mask)
        logits = self.lm_head(self.final_norm(hidden))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

model = CharacterGPT(vocab_size=vocab_size, seq_len=SEQ_LEN).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count:,}")
print(f"Device: {device}")

C:\Users\spide\AppData\Local\Temp\ipykernel_13840\1306350586.py:22: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(block, num_layers=num_layers)


Parameters: 3,230,464
Device: cuda


In [10]:
from pathlib import Path

checkpoint_path = Path("character_gpt_best.pt")
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded checkpoint from step {checkpoint['step']} with validation loss {checkpoint['val_loss']:.4f}")
else:
    print("No checkpoint found; use the training cell before generating text.")

Loaded checkpoint from step 5000 with validation loss 1.3652


In [12]:
# Smoke test: verify shapes and confirm the causal model produces a scalar loss.
example_inputs, example_targets = next(iter(train_dl))
example_inputs = example_inputs.to(device)
example_targets = example_targets.to(device)
logits, loss = model(example_inputs, example_targets)
print(f"Input shape: {tuple(example_inputs.shape)}")
print(f"Logits shape: {tuple(logits.shape)}")
print(f"Initial loss: {loss.item():.4f} (random baseline is about {torch.log(torch.tensor(vocab_size, dtype=torch.float32)).item():.4f})")

Input shape: (64, 128)
Logits shape: (64, 128, 149)
Initial loss: 1.4011 (random baseline is about 5.0039)


### Training

We minimize cross-entropy between the predicted next-character distribution and the actual next character. AdamW updates the parameters, while validation loss tells us whether the model is learning general patterns instead of only memorizing training sequences.

In [22]:
import math
import time

learning_rate = 3e-4
weight_decay = 0.1
max_steps = 5000
eval_interval = 100
eval_batches = 20

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}
    for name, loader in (("train", train_loader), ("val", val_loader)):
        losses = []
        loader_iterator = iter(loader)
        for _ in range(eval_batches):
            try:
                inputs, targets = next(loader_iterator)
            except StopIteration:
                break
            _, batch_loss = model(inputs.to(device), targets.to(device))
            losses.append(batch_loss.item())
        results[name] = sum(losses) / len(losses)
    model.train()
    return results

best_val_loss = math.inf
train_iterator = iter(train_loader)
training_start = time.time()

for step in range(1, max_steps + 1):
    try:
        inputs, targets = next(train_iterator)
    except StopIteration:
        train_iterator = iter(train_loader)
        inputs, targets = next(train_iterator)

    _, loss = model(inputs.to(device), targets.to(device))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step == 1 or step % eval_interval == 0:
        losses = estimate_loss()
        elapsed = time.time() - training_start
        print(
            f"step {step:>4}/{max_steps} | "
            f"train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f} | "
            f"elapsed {elapsed:.1f}s"
        )
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "step": step,
                    "val_loss": best_val_loss,
                    "char2int": char2int,
                    "chars_sorted": chars_sorted,
                },
                "character_gpt_best.pt",
            )
            print("  saved character_gpt_best.pt")

step    1/5000 | train loss 4.5627 | val loss 4.5380 | elapsed 2.3s
  saved character_gpt_best.pt
step  100/5000 | train loss 2.6024 | val loss 2.5429 | elapsed 8.4s
  saved character_gpt_best.pt
step  200/5000 | train loss 2.4874 | val loss 2.4533 | elapsed 14.3s
  saved character_gpt_best.pt
step  300/5000 | train loss 2.3041 | val loss 2.2553 | elapsed 20.4s
  saved character_gpt_best.pt
step  400/5000 | train loss 2.1230 | val loss 2.1058 | elapsed 26.4s
  saved character_gpt_best.pt
step  500/5000 | train loss 2.0180 | val loss 1.9979 | elapsed 32.2s
  saved character_gpt_best.pt
step  600/5000 | train loss 1.9225 | val loss 1.9255 | elapsed 38.2s
  saved character_gpt_best.pt
step  700/5000 | train loss 1.8263 | val loss 1.8672 | elapsed 44.2s
  saved character_gpt_best.pt
step  800/5000 | train loss 1.7768 | val loss 1.8256 | elapsed 50.2s
  saved character_gpt_best.pt
step  900/5000 | train loss 1.7290 | val loss 1.7929 | elapsed 56.4s
  saved character_gpt_best.pt
step 1000/50

### Generate text

At inference time there is no target sequence. The model predicts a probability distribution for the next character, samples from it, appends that character, and repeats. `temperature` controls randomness: lower values are more predictable, while higher values are more diverse.

In [23]:
import torch.nn.functional as F

@torch.no_grad()
def generate_text(model, prompt, max_new_chars=500, temperature=0.8):
    if not prompt:
        raise ValueError("prompt must contain at least one character")
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")

    missing_characters = sorted(set(prompt) - set(char2int))
    if missing_characters:
        raise ValueError(f"Prompt contains characters outside the vocabulary: {missing_characters}")

    model.eval()
    model_device = next(model.parameters()).device
    encoded_prompt = [char2int[character] for character in prompt]
    generated = torch.tensor(encoded_prompt, dtype=torch.long).unsqueeze(0).to(model_device)

    for _ in range(max_new_chars):
        context = generated[:, -SEQ_LEN:]
        logits, _ = model(context)
        next_token_logits = logits[:, -1, :] / temperature
        probabilities = F.softmax(next_token_logits, dim=-1)
        probabilities = torch.nan_to_num(probabilities, nan=0.0, posinf=0.0, neginf=0.0)
        probabilities = probabilities / probabilities.sum(dim=-1, keepdim=True)
        next_token = torch.multinomial(probabilities.cpu(), num_samples=1).to(model_device)
        generated = torch.cat((generated, next_token), dim=1)

    generated_text= "".join(chars_sorted[index] for index in generated[0].cpu().tolist())
    
    import time
    for char in generated_text:
        print(char, end='', flush=True)
        time.sleep(0.03)  # Adjust the sleep time for desired speed
        


prompt = "Power "
generate_text(model, prompt, max_new_chars=300, temperature=0.8)

Power were
the popularly women had been been beginning to say the Athenians to treat
attention to Similar Athens had discovered himself as they now only has
ever contrast as long values of the party. It was all of the word and affective
feelings of her unconsciously. Mary Instead a stablish would triumpen

In [24]:
# Okay, Let's try a lower temperature and let the model do its wonders (maybe?)
prompt = "Both laws: 'Never outshine the master' and 'Always be an object of desire' may actually have something in common which is"
generate_text(model, prompt, max_new_chars=600, temperature=0.5)

Both laws: 'Never outshine the master' and 'Always be an object of desire' may actually have something in common which is
all ways to see the same years. We can be say the best designed of the bold
of our mother teams. We are not the conventional moment of the world that is
the constant state of what we are strategistic and control of the strong
men of constant the power we can be able to be the relationship of the
show and interviews about the children, he would be not failed and out
of the strategy was a strategy of the most of the most prince would be impossible
to seduce him on the prone of the most problems of the courtiers and the
easy to his voice. He also would seem that the next sense of the hands of
hi

In [25]:
prompt= "Life is not a game to be attended by our real faces, those who remain naive will "
generate_text(model, prompt, max_new_chars=500, temperature=0.4)

Life is not a game to be attended by our real faces, those who remain naive will be
served of the new way. In the end of the most people with their prince, and
they were all the countess of the contraction of the strategy, and they may
the more people than they were some of the past and control. They were so
much more than they would have the think of the most problem of their
sense of desires and the strategy that we can be seen the world that we want
to interest. We are best to be a terrifying of the group we find our
truth and control our enemies and control. We are the c